# BassSpecMatchPRO — NAM Colab
Execute a célula abaixo. Ela inicia o worker temporário; o app envia automaticamente o job **Reference-only**.


In [ ]:
# BassSpecMatchPRO public bootstrap — no persistent credentials
import os, sys, subprocess, secrets, json, time, threading, zipfile, shutil, re, urllib.request
from pathlib import Path
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'flask'])
from flask import Flask, request, jsonify, send_file
CLOUDFLARED='/content/cloudflared'
if not (os.path.isfile(CLOUDFLARED) and os.access(CLOUDFLARED, os.X_OK)):
    tmp=CLOUDFLARED+'.download-'+secrets.token_hex(4)
    try:
        urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', tmp)
        os.chmod(tmp, 0o755)
        os.replace(tmp, CLOUDFLARED)
    finally:
        if os.path.exists(tmp): os.remove(tmp)
ROOT=Path('/content/bassspec_nam_worker'); ROOT.mkdir(parents=True, exist_ok=True)
TOKEN=secrets.token_urlsafe(32); state={'state':'ready','phase':'waiting','epoch':0,'epochs':0,'error':''}
app=Flask(__name__)
def auth(): return request.headers.get('Authorization','') == 'Bearer '+TOKEN
@app.get('/health')
def health(): return jsonify({'ok':True}) if auth() else ('Unauthorized',401)
@app.post('/job')
def job():
    if not auth(): return ('Unauthorized',401)
    z=ROOT/'job.zip'; z.write_bytes(request.get_data()); state.update(state='received',phase='queued'); return jsonify({'ok':True})
@app.get('/status')
def status(): return jsonify(state) if auth() else ('Unauthorized',401)
@app.get('/result')
def result():
    if not auth(): return ('Unauthorized',401)
    p=ROOT/'trained.nam'; return send_file(p,as_attachment=True) if p.exists() else ('Not ready',404)
def train_loop():
    while True:
        z=ROOT/'job.zip'
        if z.exists() and state['state']=='received':
            try:
                work=ROOT/'job'; shutil.rmtree(work,ignore_errors=True); work.mkdir()
                with zipfile.ZipFile(z) as f: f.extractall(work)
                manifest=json.loads((work/'manifest.json').read_text())
                assert manifest.get('mode') in ('reference-only','reference_only') or 'source' not in json.dumps(manifest).lower()
                epochs=int(manifest.get('epochs',25)); state.update(state='training',phase='training',epochs=epochs)
                cmd=[sys.executable,str(work/'train_nam_official.py'),'--reference',str(work/'reference.wav'),'--template',str(work/'model.nam'),'--output',str(ROOT/'trained.nam'),'--epochs',str(epochs)]
                proc=subprocess.Popen(cmd,cwd=work)
                progress=work/'training-progress.json'
                while proc.poll() is None:
                    if progress.exists():
                        try:
                            d=json.loads(progress.read_text()); state.update(phase=d.get('phase','training'),epoch=int(d.get('epoch',0)),epochs=int(d.get('epochs',epochs)))
                        except Exception: pass
                    time.sleep(1)
                if proc.returncode: raise RuntimeError('trainer failed: '+str(proc.returncode))
                state.update(state='complete',phase='complete',epoch=epochs,epochs=epochs)
            except Exception as e: state.update(state='error',phase='error',error=str(e))
        time.sleep(.5)
threading.Thread(target=train_loop,daemon=True).start()
threading.Thread(target=lambda: app.run(host='127.0.0.1',port=5000,use_reloader=False),daemon=True).start()
time.sleep(1)
tunnel=subprocess.Popen([CLOUDFLARED,'tunnel','--url','http://127.0.0.1:5000','--no-autoupdate'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
public=None; deadline=time.time()+45
while time.time()<deadline:
    line=tunnel.stdout.readline()
    if not line:
        if tunnel.poll() is not None: break
        time.sleep(.1); continue
    m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',line)
    if m: public=m.group(0); break
if not public:
    tunnel.terminate(); raise RuntimeError('Cloudflare Quick Tunnel não iniciou; execute a célula novamente.')
print('BASSSPEC_COLAB='+public+'|'+TOKEN, flush=True)
print('Worker pronto. Volte ao BassSpecMatchPRO; o app continuará automaticamente após receber essa linha.')
